# 3. Embeddings & Vector Stores (Lưu trữ Vector)

## 📖 Lý thuyết
- **Embeddings**: Biến đổi văn bản thành một danh sách các số thực (Vector). Các câu có ý nghĩa giống nhau sẽ có vector nằm gần nhau trong không gian.
- **Vector Store**: Cơ sở dữ liệu chuyên dụng để lưu trữ các Vector này và cho phép tìm kiếm cực kỳ nhanh (Similarity Search). Trong RAG SGU, bạn dùng **FAISS** (Facebook AI Similarity Search).

## 💻 Code mẫu
Khởi tạo HuggingFace model nhúng text và lưu vào FAISS (Mô phỏng `vector_store.py`).

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Lưu ý: LangChain đã tách module sang thư viện core, hỗ trợ cảnh báo
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except:
    from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Tạo dữ liệu text mẫu cơ bản (thay vì dùng PDF)
sample_text = """
Ngành Công nghệ Thông tin (CNTT) tại trường Đại học Sài Gòn (SGU) đào tạo sinh viên 
trở thành các kỹ sư phần mềm, chuyên gia mạng máy tính và hệ thống thông tin.
Mục tiêu đào tạo là cung cấp kiến thức nền tảng vững chắc và kỹ năng thực hành chuyên sâu.
Chuẩn đầu ra của ngành bao gồm khả năng lập trình, phân tích thiết kế hệ thống và phát triển ứng dụng AI.
"""

# 2. Chia nhỏ text thành chunks (Tạo LangChain Document)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20,
    separators=["\n", ".", " ", ""]
)
docs = splitter.create_documents([sample_text])
print(f"Đã tách đoạn mã thành {len(docs)} chunks.")

# 3. Khởi tạo Embedding model
model_name = "paraphrase-multilingual-mpnet-base-v2"
print(f"Đang tải mô hình Embedding: {model_name}... (có thể tốn chút thời gian)")
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    encode_kwargs={"normalize_embeddings": True}  # Cấu hình chuẩn hóa vector
)

# 4. Đưa chunks vào FAISS
print("Tạo FAISS Vector Store từ dữ liệu text cơ bản...")
vector_store = FAISS.from_documents(docs, embeddings)

# 5. Thử tìm kiếm
query = "Mục tiêu đào tạo và chuẩn đầu ra của ngành CNTT là gì?"
print(f"\nCâu hỏi: '{query}'")
results = vector_store.similarity_search(query, k=2)

print("\n--- Kết quả tìm kiếm gần nhất ---")
for i, res in enumerate(results):
    print(f"Kết quả {i+1}:", res.page_content.strip())


a:\sgu-student-rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đã tách đoạn mã thành 4 chunks.
Đang tải mô hình Embedding: paraphrase-multilingual-mpnet-base-v2... (có thể tốn chút thời gian)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3386.81it/s]
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tạo FAISS Vector Store từ dữ liệu text cơ bản...

Câu hỏi: 'Mục tiêu đào tạo và chuẩn đầu ra của ngành CNTT là gì?'

--- Kết quả tìm kiếm gần nhất ---
Kết quả 1: Ngành Công nghệ Thông tin (CNTT) tại trường Đại học Sài Gòn (SGU) đào tạo sinh viên
Kết quả 2: Mục tiêu đào tạo là cung cấp kiến thức nền tảng vững chắc và kỹ năng thực hành chuyên sâu.


## 🎯 Bài tập nhỏ
**Yêu cầu**: Tìm cách lưu FAISS index này vào một thư mục có tên `faiss_test_index` và sau đó viết một dòng code load FAISS index đó lên lại vào một biến tên là `loaded_vector_store`.

In [2]:
from langchain_community.vectorstores import FAISS

# 1. Lưu index xuống ổ cứng thư mục 'faiss_test_index'
save_path = "faiss_test_index"
print(f"Đang lưu FAISS index vào thư mục '{save_path}'...")
vector_store.save_local(save_path)
print("=> Đã lưu thành công.\n")

# 2. Tải index lên lại vào biến loaded_vector_store
print("Đang tải FAISS index...")
loaded_vector_store = FAISS.load_local(
    save_path, 
    embeddings, 
    allow_dangerous_deserialization=True
)

# 3. Test thử bản load được
reload_query = "Mục tiêu đào tạo và chuẩn đầu ra của ngành CNTT là gì?"
print(f"=> Đã tải thành công! Test lại với câu hỏi: '{reload_query}'")
res = loaded_vector_store.similarity_search(reload_query, k=1)
print(res[0].page_content[:800])
print("\nNguồn:", res[0].metadata.get("source", "N/A"))
print("Trang:", res[0].metadata.get("page", "N/A"))


Đang lưu FAISS index vào thư mục 'faiss_test_index'...
=> Đã lưu thành công.

Đang tải FAISS index...
=> Đã tải thành công! Test lại với câu hỏi: 'Mục tiêu đào tạo và chuẩn đầu ra của ngành CNTT là gì?'
Ngành Công nghệ Thông tin (CNTT) tại trường Đại học Sài Gòn (SGU) đào tạo sinh viên

Nguồn: N/A
Trang: N/A
